### A/B Testing

The applied version of `probability-statistics.ipynb`'s hypothesis-testing foundations (CLT, confidence intervals), specifically for comparing two versions of something (a model, a UI, a policy) via a randomized controlled experiment.

#### 0. Core idea

Randomly split units (users, transactions) into a control group (existing version, A) and a treatment group (new version, B). Compare a metric between them. Randomization is what makes the comparison valid, it is what allows attributing any observed difference to the change itself, not to some other systematic difference between the two groups (e.g. if treatment users happened to be assigned all the high-value customers, any lift would be confounded, not caused by the treatment).

General hypothesis-testing framework (H0/H1, test statistic, Type I/II error) covered in `probability-statistics.ipynb`'s Hypothesis Testing section, not repeated here. This notebook is that framework applied specifically to the two-group comparison case, most commonly a two-proportion test (conversion rates, catch rates), a two-sample t-test would be the right tool instead if comparing MEANS rather than proportions (also covered in `probability-statistics.ipynb`).

#### 1. Two-proportion z-test, worked by hand

Toy setup: current fraud model (A, control) catches fraud in 120 of 1000 reviewed transactions (12%). A new model (B, treatment) catches 140 of 1000 (14%). Is the improvement real, or could it be noise?
```
p1 = 120/1000 = 0.12,  p2 = 140/1000 = 0.14
pooled p = (120+140) / (1000+1000) = 260/2000 = 0.13

standard error = sqrt(pooled_p*(1-pooled_p) * (1/n1 + 1/n2))
               = sqrt(0.13*0.87 * (1/1000 + 1/1000))
               = sqrt(0.1131 * 0.002)
               = sqrt(0.0002262)
               = 0.0150

z = (p2 - p1) / standard_error = (0.14-0.12) / 0.0150 = 1.33
```
z=1.33 corresponds to a two-tailed p-value of about 0.184, well above the standard 0.05 significance threshold. Despite a real-looking 2-point absolute improvement (12% to 14%, a 16.7% relative lift), the sample size (1000 per group) is not large enough to confidently rule out that this gap is just noise. This is a genuinely common, realistic outcome, not a contrived example, real improvements often look promising in raw numbers but fail to clear statistical significance at typical sample sizes.

In [ ]:
import numpy as np
from scipy import stats

n1, n2 = 1000, 1000
successes1, successes2 = 120, 140
p1, p2 = successes1 / n1, successes2 / n2

pooled_p = (successes1 + successes2) / (n1 + n2)
se = np.sqrt(pooled_p * (1 - pooled_p) * (1/n1 + 1/n2))
z = (p2 - p1) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z)))  # two-tailed

print(f"p1={p1}, p2={p2}, z={z:.3f}, p-value={p_value:.3f}")
print("significant at alpha=0.05:", p_value < 0.05)

#### 2. What the p-value actually means, and the common misreading

p-value = the probability of observing a difference this large or larger, IF the null hypothesis were true (IF there were actually no real difference). It is NOT the probability that H0 is true, and it is NOT the probability that the observed effect is real, a genuinely common misreading. A p-value of 0.184 above means: "if A and B were truly identical, there is an 18.4% chance random sampling alone would produce a gap this big or bigger," not "there's an 18.4% chance B is actually better."

#### 3. Statistical significance vs. practical significance

A large enough sample size can make a TINY, practically meaningless difference statistically significant (p < 0.05), and a real, practically important difference can fail to reach significance with too small a sample (exactly what happened above). Always report the effect size (here, the 2-point absolute lift) alongside the p-value, not the p-value alone, a "significant" result with a trivial effect size may not be worth shipping, and a "not significant" result with a meaningful effect size may just mean the test needs more data, not that the change does not work.

#### 4. Sample size / power, worked

Power = probability of correctly detecting a real effect when one exists (1 - probability of a false negative). Underpowered tests (too few samples) are the actual root cause of the result above, the true effect might be real, but 1000 per group was not enough to detect a 2-point gap reliably.

How many samples would be needed to reliably detect this same 2-point gap (12% vs 14%), at 80% power and 5% significance?

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

effect_size = proportion_effectsize(0.14, 0.12)
power_analysis = NormalIndPower()
required_n = power_analysis.solve_power(effect_size=effect_size, alpha=0.05, power=0.8, ratio=1.0)

print("effect size (Cohen's h):", round(effect_size, 4))
print("required sample size PER GROUP for 80% power:", round(required_n))

#### 5. Common pitfalls

Peeking: checking the p-value repeatedly as data accumulates and stopping as soon as it crosses 0.05, inflates the false positive rate far above the nominal 5%, since you are effectively running many tests (one per peek) and only reporting the lucky one. Fix: decide the sample size upfront (via the power calculation above) and do not stop early, or use a sequential testing method designed to allow peeking honestly.

Multiple testing problem: running many simultaneous A/B tests (or testing many metrics within one experiment) and reporting whichever one happened to reach significance, the same inflation problem as peeking, just spread across tests instead of across time. Fix: correct the significance threshold for the number of tests run (Bonferroni correction divides alpha by the number of tests, the simplest, most conservative fix).

Novelty effect: a new version gets an initial lift just because it is NEW and users are curious/paying more attention, not because it is genuinely better, the lift fades once the novelty wears off. Fix: run the test long enough to look past the initial novelty window, and watch for a declining effect size over the test's duration as a warning sign.

Improper randomization: if the split is not truly random (e.g. by day of week, or any process correlated with the outcome), the two groups can differ systematically before the treatment is even applied, breaking the causal interpretation entirely, the entire point of randomization.